# 🏋️ Azure Search Service (Foundry IQ) + Foundry Agent: Fitness-Fun Workshop 🤸

Welcome to this self-guided workshop where you'll:

1. Create an Azure Search Service (Foundry IQ) index containing some sample fitness equipment data.
2. Upload and verify your documents.
3. Create a **Foundry Agent** (Prompt Agent) with the native **Azure Search Service (Foundry IQ) tool**, so the agent grounds its own answers in that index.
4. Run a multi-turn conversation to query your data (with a fun fitness twist).

This notebook follows the newer Foundry endpoint-based flow and does not require downgrading `azure-ai-projects`.

> **Note:** an earlier version of this notebook manually retrieved search results in Python and pasted them into the prompt by hand. This version instead attaches Azure AI Search **directly to the agent as a tool** — the agent handles retrieval internally, which is the current recommended pattern (see `4-ai_search.ipynb` for the same approach applied to the identical index).

Also ensure you've set these environment variables:

- `PROJECT_ENDPOINT`
- `MODEL_DEPLOYMENT_NAME`
- `SEARCH_ENDPOINT`, `SEARCH_API_KEY` (for creating and populating the index)
- `SEARCH_CONNECTION_NAME` (the name of the Azure AI Search connection in your Foundry project, created under Operate → Admin → Connected resources)

Let's get started!

## Prerequisites

Before running the cells below, please verify:
- You installed the workshop requirements with stable Foundry v2 packages.
- Your environment is configured with `PROJECT_ENDPOINT`, `MODEL_DEPLOYMENT_NAME`, `SEARCH_ENDPOINT`, `SEARCH_API_KEY`, and `SEARCH_CONNECTION_NAME`.

## 1. Create & Populate Azure Search Service (Foundry IQ) Index

In this section we will:
- Create an Azure Search Service (Foundry IQ) index called `myfitnessindex` with a schema suited for fitness items
- Upload sample documents containing fitness equipment data
- Verify that the documents are searchable

Make sure your environment has the appropriate search credentials (typically obtained via your Foundry project).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.search.documents import SearchClient

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

# Azure AI Search endpoint + admin key (from the Search resource in the Azure portal)
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_credential = AzureKeyCredential(os.environ["SEARCH_API_KEY"])

# Define the index name for our fitness data
index_name = "myfitnessindex"

try:
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
    print("✅ Created SearchIndexClient")

    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )
    print("✅ Created SearchClient for document operations")
except Exception as e:
    print(f"❌ Error creating search clients: {e}")

✅ Created SearchIndexClient
✅ Created SearchClient for document operations


### Define the Index Schema

We will create an index with the following fields:
- `FitnessItemID`: Unique key
- `Name`: Searchable text field (also filterable)
- `Category`: Searchable, filterable, and facetable (e.g. Strength, Cardio, Flexibility)
- `Price`: Numeric field (filterable, sortable, and facetable)
- `Description`: Full-text searchable field

In [2]:
def create_fitness_index():
    fields = [
        SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
        SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
        SearchableField(name="Description", type=SearchFieldDataType.String)
    ]

    index = SearchIndex(name=index_name, fields=fields)

    # Delete the index if it already exists (for a fresh start)
    if index_name in [x.name for x in index_client.list_indexes()]:
        index_client.delete_index(index_name)
        print(f"🗑️ Deleted existing index: {index_name}")

    created = index_client.create_index(index)
    print(f"🎉 Created index: {created.name}")

create_fitness_index()

🎉 Created index: myfitnessindex


### Upload Sample Documents

Now we'll add some sample fitness items to `myfitnessindex`.

In [3]:
def upload_fitness_docs():
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )

    sample_docs = [
        {
            "FitnessItemID": "1",
            "Name": "Adjustable Dumbbell",
            "Category": "Strength",
            "Price": 59.99,
            "Description": "A compact, adjustable weight for targeted muscle workouts."
        },
        {
            "FitnessItemID": "2",
            "Name": "Yoga Mat",
            "Category": "Flexibility",
            "Price": 25.0,
            "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
        },
        {
            "FitnessItemID": "3",
            "Name": "Treadmill",
            "Category": "Cardio",
            "Price": 499.0,
            "Description": "A sturdy treadmill with adjustable speed and incline settings."
        },
        {
            "FitnessItemID": "4",
            "Name": "Resistance Bands",
            "Category": "Strength",
            "Price": 15.0,
            "Description": "Set of colorful bands for light to moderate resistance workouts."
        }
    ]

    result = search_client.upload_documents(documents=sample_docs)
    print(f"🚀 Upload result: {result}")

upload_fitness_docs()
print("✅ Documents uploaded to search index")

🚀 Upload result: [<azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000014CE8CF4AA0>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000014CE8CF4B00>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000014CE8CF4AD0>, <azure.search.documents._generated.models._models_py3.IndexingResult object at 0x0000014CE8CF4B30>]
✅ Documents uploaded to search index


### Verify the Documents

Let's perform a basic search query (e.g. for items in the Strength category) to ensure everything is working before we hand this index to an agent.

In [4]:
results = search_client.search(search_text="Strength", filter=None, top=10)

print("🔍 Search results for 'Strength':")
print("-" * 50)
found_items = False
for doc in results:
    found_items = True
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 50)

if not found_items:
    print("No matching items found.")

🔍 Search results for 'Strength':
--------------------------------------------------
Name: Resistance Bands
Category: Strength
Price: $15.00
Description: Set of colorful bands for light to moderate resistance workouts.
--------------------------------------------------
Name: Adjustable Dumbbell
Category: Strength
Price: $59.99
Description: A compact, adjustable weight for targeted muscle workouts.
--------------------------------------------------


## 2. Create a Foundry Agent with the Azure Search Service (Foundry IQ) Tool

In this section we create a **Foundry Prompt Agent** and attach the **Azure Search Service (Foundry IQ) tool** directly to it, pointed at `myfitnessindex`. The agent handles retrieval internally — we no longer manually query the index in Python and paste results into the prompt.

The agent:
- Runs on your model deployment (`MODEL_DEPLOYMENT_NAME`).
- Searches `myfitnessindex` itself via the Azure Search Service (Foundry IQ) tool whenever it needs product information.
- Holds a multi-turn conversation using the Responses API conversation object.

In [5]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
    ConnectionType,
)

# New Foundry uses the project endpoint (no connection string).
project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_deployment_name = os.environ["MODEL_DEPLOYMENT_NAME"]
search_connection_name = os.environ.get("SEARCH_CONNECTION_NAME")

if not model_deployment_name:
    raise ValueError("MODEL_DEPLOYMENT_NAME not set in .env")
if not project_endpoint:
    raise ValueError("PROJECT_ENDPOINT not set in .env")

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

# Resolve the Azure AI Search connection ID. Connection names in Foundry often
# carry a random suffix, so we resolve by type unless an exact name is given.
connection = None
if search_connection_name:
    try:
        connection = project_client.connections.get(search_connection_name)
    except Exception:
        connection = None

if connection is None:
    for conn in project_client.connections.list():
        if conn.type == ConnectionType.AZURE_AI_SEARCH:
            connection = conn
            break

if connection is None:
    raise RuntimeError(
        "No Azure AI Search connection found in this project. "
        "Add one under Operate > Admin > Connected resources."
    )

print(f"Using Azure AI Search connection '{connection.name}'")
search_connection_id = connection.id

agent = None
conversation = None

try:
    # Build the Azure AI Search tool pointing at our index.
    # query_type=SIMPLE runs keyword search (our index has no vector field).
    ai_search_tool = AzureAISearchTool(
        azure_ai_search=AzureAISearchToolResource(
            indexes=[
                AISearchIndexResource(
                    project_connection_id=search_connection_id,
                    index_name=index_name,
                    query_type=AzureAISearchQueryType.SIMPLE,
                )
            ]
        )
    )

    # Create a Foundry prompt agent for fitness shopping assistance, with the
    # search tool attached directly - no manual retrieval step needed.
    agent = project_client.agents.create_version(
        agent_name="fitness-shopping-assistant",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
                You are a Fitness Shopping Assistant. Use the search tool to find relevant
                fitness equipment from the catalog. Cite the item names you used, and always
                add a short disclaimer that you are not providing medical advice.
            """,
            tools=[ai_search_tool],
        ),
        description="Fitness shopping assistant grounded on the Azure AI Search index.",
    )
    print(f"🎉 Created agent '{agent.name}', version: {agent.version}")

    conversation = openai_client.conversations.create()

    user_queries = [
        "Which items are best for strength training?",
        "I need something for cardio under $300. Any suggestions?",
    ]

    for query in user_queries:
        print(f"\n# User: {query}\n")
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=query,
            tool_choice="required",
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"# Agent: {response.output_text}\n")
finally:
    if agent and project_client:
        project_client.agents.delete_version(
            agent_name=agent.name,
            agent_version=agent.version,
        )
        print("🗑️ Cleaned up agent version")

Using Azure AI Search connection 'aisearch2346625rdg92y'
🎉 Created agent 'fitness-shopping-assistant', version: 1

# User: Which items are best for strength training?

# Agent: For strength training, the best options from the catalog are:

- **Adjustable Dumbbell** — great for progressive overload and targeted muscle workouts; usually the most versatile pick for building strength at home【5:1†source】
- **Resistance Bands** — useful for light to moderate resistance work, accessory exercises, warm-ups, and portability【5:0†source】

If your goal is overall strength, I’d rank them:
1. **Adjustable Dumbbell**【5:1†source】
2. **Resistance Bands**【5:0†source】

If you want, I can also recommend the best item based on your space, budget, or beginner/intermediate level.

Not medical advice.


# User: I need something for cardio under $300. Any suggestions?

# Agent: For cardio under $300, I only found **Treadmill** in the catalog results, listed under the **Cardio** category with adjustable speed a

## 3. Cleanup

For this demo we already clean up the agent version inside the previous cell. In case you want to remove the search index as well (for a fresh start), run the cell below.

In [7]:
try:
    index_client.delete_index(index_name)
    print(f"🗑️ Deleted index {index_name}")
except Exception as e:
    print(f"Error deleting index: {e}")

🗑️ Deleted index myfitnessindex


## 🎉 Congrats!

You've successfully:
- Created an Azure Search Service (Foundry IQ) index and populated it with fitness data.
- Verified the data via a basic search query.
- Built and ran a **Foundry Agent** with the **Azure Search Service (Foundry IQ) tool** attached directly, so the agent handles retrieval internally rather than through manual, hand-written search calls.

Feel free to explore further enhancements, such as integrating additional tools, evaluation, and tracing workflows in Microsoft Foundry.